# ⚡ PowerBlind+ : Google Colab Execution Notebook
### Explainable Multi-Class Smart Meter Anomaly Diagnosis & Decision Support System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**PowerBlind+** detects and explains electricity theft, faulty meters, and communication failures in smart meter data using **XGBoost** and **SHAP** values.

---

### Pipeline Steps:
1. **Hardware & Environment Check** (GPU/CPU acceleration)
2. **Install ML Dependencies** (xgboost, shap, scikit-learn, imbalanced-learn, fastapi)
3. **Load or Generate Dataset** (5,000 consumers × 365 days = 1.825M readings)
4. **Feature Engineering** (26 electrical and statistical features per consumer)
5. **Model Training with SMOTE** (Stratified 5-Fold Cross Validation + Final XGBoost model)
6. **Performance Evaluation & Confusion Matrix Heatmap**
7. **Explainable AI (XAI) with SHAP** (Global feature importance & local Waterfall plots)
8. **Export & Download Trained Artifacts** (models/model_latest.joblib, metrics.json, shap.json)
9. **Run Live FastAPI Microservice** & Interactive Diagnosis Widget


## 1. Hardware & Environment Check


In [ ]:
import platform, os
print('Python Version:', platform.python_version())
print('OS Platform:   ', platform.platform())

try:
    import torch
    if torch.cuda.is_available():
        print(f'CUDA GPU Available: True ({torch.cuda.get_device_name(0)})')
    else:
        print('CUDA GPU Available: False (Running on CPU)')
except Exception as e:
    print('GPU check status:', e)


## 2. Install ML Dependencies


In [ ]:
!pip install -q xgboost shap scikit-learn imbalanced-learn fastapi uvicorn pydantic pandas numpy joblib matplotlib seaborn
print('Dependencies installed successfully!')


## 3. Workspace & Directory Setup
Automatically configures the paths whether running inside a cloned repository or uploaded folder.


In [ ]:
import os, sys, glob, zipfile
from pathlib import Path

# 1. Automatic Zip Detection & Extraction
zip_candidates = glob.glob('/content/*.zip') + glob.glob('*.zip')
for zpath in zip_candidates:
    zname = os.path.basename(zpath).lower()
    if 'powerblind' in zname or 'dataset' in zname:
        print(f"📦 Found uploaded archive: {zpath}")
        try:
            with zipfile.ZipFile(zpath, 'r') as zf:
                top_members = {item.split('/')[0] for item in zf.namelist() if '/' in item}
                if 'powerblind-plus' in top_members:
                    print("Unzipping archive to /content ...")
                    zf.extractall('/content')
                else:
                    print("Unzipping archive to /content/powerblind-plus ...")
                    zf.extractall('/content/powerblind-plus')
            print(" Archive extracted successfully!")
        except Exception as e:
            print(f"Extraction warning: {e}")
        break

# 2. Determine REPO_ROOT
possible_roots = [
    Path('/content/powerblind-plus'),
    Path('.').resolve(),
    Path('/content')
]
REPO_ROOT = Path('/content')
for pr in possible_roots:
    if (pr / 'services').exists() or (pr / 'data').exists() or (pr / 'PowerBlind_Plus_Dataset').exists():
        REPO_ROOT = pr
        break

if REPO_ROOT.name == 'powerblind-plus' and os.path.exists('/content/powerblind-plus'):
    %cd /content/powerblind-plus
    REPO_ROOT = Path('/content/powerblind-plus')

# 3. Locate Dataset (check data/ or PowerBlind_Plus_Dataset/)
possible_data_dirs = [
    REPO_ROOT / 'data',
    REPO_ROOT / 'PowerBlind_Plus_Dataset',
    Path('/content/powerblind-plus/data'),
    Path('/content/powerblind-plus/PowerBlind_Plus_Dataset'),
    Path('/content/data'),
    Path('/content/PowerBlind_Plus_Dataset')
]
DATA_DIR = REPO_ROOT / 'data'
for dd in possible_data_dirs:
    if (dd / 'powerblind_smartmeter_readings.csv').exists():
        DATA_DIR = dd
        break

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = REPO_ROOT / 'services' / 'ml' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

READINGS_CSV = DATA_DIR / 'powerblind_smartmeter_readings.csv'
LABELS_CSV = DATA_DIR / 'powerblind_consumer_labels.csv'

print(f'Repo root:    {REPO_ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Models dir:   {MODELS_DIR}')
print(f'Readings CSV exists: {READINGS_CSV.exists()}')
print(f'Labels CSV exists:   {LABELS_CSV.exists()}')


### 3.1 Check or Generate Dataset
If powerblind_smartmeter_readings.csv is not detected, it will be generated automatically.


In [ ]:
import os, sys
from pathlib import Path

if not READINGS_CSV.exists() or not LABELS_CSV.exists():
    print('Dataset CSVs not detected. Generating synthetic 5,000-consumer dataset (~1-2 min)...')
    possible_gen_scripts = [
        DATA_DIR / 'generate_dataset.py',
        REPO_ROOT / 'data' / 'generate_dataset.py',
        REPO_ROOT / 'PowerBlind_Plus_Dataset' / 'generate_dataset.py',
        Path('/content/powerblind-plus/data/generate_dataset.py'),
    ]
    gen_script = None
    for pgs in possible_gen_scripts:
        if pgs.exists():
            gen_script = pgs
            break
            
    if gen_script:
        os.system(f'{sys.executable} {gen_script}')
        for dd in [DATA_DIR, REPO_ROOT / 'data', REPO_ROOT / 'PowerBlind_Plus_Dataset']:
            if (dd / 'powerblind_smartmeter_readings.csv').exists():
                READINGS_CSV = dd / 'powerblind_smartmeter_readings.csv'
                LABELS_CSV = dd / 'powerblind_consumer_labels.csv'
                DATA_DIR = dd
                break
    else:
        print('Please ensure data/generate_dataset.py or PowerBlind_Plus_Dataset/generate_dataset.py is available.')
else:
    print(f'Dataset CSVs are present and ready in {DATA_DIR}!')


## 4. Dataset Inspection & Time-Series Visualizations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

labels_df = pd.read_csv(LABELS_CSV)
print('=== Ground Truth Consumer Class Counts ===')
print(labels_df['label'].value_counts())
print('')
print('=== Anomaly Subtypes ===')
print(labels_df['anomaly_subtype'].value_counts().head(10))

sample_readings = pd.read_csv(READINGS_CSV, nrows=50000, low_memory=False)
display(sample_readings.head())

plt.figure(figsize=(14, 8))
classes = ['Normal', 'Theft', 'FaultyMeter', 'CommFailure']
colors = ['#22c55e', '#ef4444', '#f59e0b', '#3b82f6']

for i, cls in enumerate(classes):
    cid_matches = labels_df[labels_df['label'] == cls]['consumer_id']
    if len(cid_matches) > 0:
        cid = cid_matches.iloc[0]
        c_readings = sample_readings[sample_readings['consumer_id'] == cid].copy()
        if len(c_readings) > 0:
            plt.subplot(2, 2, i+1)
            plt.plot(pd.to_datetime(c_readings['date']), c_readings['kwh'], color=colors[i], lw=1.5)
            plt.title(f'{cls} Pattern (Consumer: {cid})', fontweight='bold')
            plt.xlabel('Date')
            plt.ylabel('Daily kWh')
            plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Feature Engineering (26 Features)
Computes 26 statistical, electrical, and temporal features per consumer from 365-day readings.


In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append(str(REPO_ROOT / 'services' / 'ml'))

try:
    from training.feature_engineering import engineer_features, FEATURE_COLS, CLASS_NAMES
    print('Imported feature engineering from services/ml/training')
except ImportError:
    FEATURE_COLS = [
        'mean_kwh', 'std_kwh', 'min_kwh', 'max_kwh',
        'kwh_range', 'kwh_iqr', 'kwh_trend_slope', 'kwh_cv',
        'mean_voltage', 'std_voltage', 'voltage_range',
        'mean_current', 'std_current',
        'mean_pf', 'std_pf', 'pf_below_09_ratio',
        'missing_ratio', 'comm_flag_ratio', 'max_missing_streak',
        'peak_to_avg_ratio', 'weekend_weekday_ratio',
        'baseline_deviation_pct',
        'type_residential_small', 'type_residential_large',
        'type_small_commercial', 'type_industrial_light',
    ]
    CLASS_NAMES = ['Normal', 'Theft', 'FaultyMeter', 'CommFailure']

    def _trend_slope(series):
        s = series.dropna()
        if len(s) < 2: return 0.0
        x = np.arange(len(s))
        return float(np.polyfit(x, s.values, 1)[0])

    def _max_streak(flags):
        max_s = cur = 0
        for v in flags:
            if v:
                cur += 1
                max_s = max(max_s, cur)
            else:
                cur = 0
        return max_s

    def engineer_features(readings_df, consumer_meta=None):
        df = readings_df.copy()
        for col in ['kwh', 'voltage_v', 'current_a', 'power_factor']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['comm_flag']  = df['comm_flag'].astype(bool)
        df['date']       = pd.to_datetime(df['date'], errors='coerce')
        df['is_weekend'] = df['date'].dt.dayofweek >= 5

        rows = []
        for cid, grp in df.groupby('consumer_id'):
            kwh  = grp['kwh']
            volt = grp['voltage_v'].dropna()
            curr = grp['current_a'].dropna()
            pf   = grp['power_factor']
            cf   = grp['comm_flag']
            n = len(grp)
            missing_mask = kwh.isna()
            kwh_valid    = kwh.dropna()
            mean_kwh = float(kwh_valid.mean()) if len(kwh_valid) else 0.0
            std_kwh  = float(kwh_valid.std())  if len(kwh_valid) > 1 else 0.0
            q75, q25 = (np.percentile(kwh_valid, [75, 25]) if len(kwh_valid) > 1 else (0.0, 0.0))
            weekend_mean = float(kwh[grp['is_weekend']].dropna().mean() or 0)
            weekday_mean = float(kwh[~grp['is_weekend']].dropna().mean() or 1e-9)
            wd_ratio     = weekend_mean / weekday_mean if weekday_mean != 0 else 1.0
            peak_avg     = float(kwh_valid.max() / mean_kwh) if mean_kwh > 0 else 1.0
            ctype = str(grp['consumer_type'].iloc[0]) if 'consumer_type' in grp.columns else ''
            base_kwh = None
            if consumer_meta is not None and cid in consumer_meta.index:
                base_kwh = float(consumer_meta.loc[cid, 'base_kwh'])
            baseline_dev = (abs(mean_kwh - base_kwh) / base_kwh if base_kwh and base_kwh > 0 else 0.0)
            rows.append({
                'consumer_id': cid, 'mean_kwh': mean_kwh, 'std_kwh': std_kwh,
                'min_kwh': float(kwh_valid.min()) if len(kwh_valid) else 0.0,
                'max_kwh': float(kwh_valid.max()) if len(kwh_valid) else 0.0,
                'kwh_range': float(kwh_valid.max() - kwh_valid.min()) if len(kwh_valid) > 1 else 0.0,
                'kwh_iqr': float(q75 - q25), 'kwh_trend_slope': _trend_slope(kwh),
                'kwh_cv': std_kwh / mean_kwh if mean_kwh > 0 else 0.0,
                'mean_voltage': float(volt.mean()) if len(volt) else 230.0,
                'std_voltage': float(volt.std()) if len(volt) > 1 else 0.0,
                'voltage_range': float(volt.max() - volt.min()) if len(volt) > 1 else 0.0,
                'mean_current': float(curr.mean()) if len(curr) else 0.0,
                'std_current': float(curr.std()) if len(curr) > 1 else 0.0,
                'mean_pf': float(pf.dropna().mean()) if pf.dropna().size else 1.0,
                'std_pf': float(pf.dropna().std()) if pf.dropna().size > 1 else 0.0,
                'pf_below_09_ratio': float((pf < 0.9).sum() / n),
                'missing_ratio': float(missing_mask.sum() / n),
                'comm_flag_ratio': float(cf.sum() / n),
                'max_missing_streak': float(_max_streak(missing_mask)),
                'peak_to_avg_ratio': float(peak_avg), 'weekend_weekday_ratio': float(wd_ratio),
                'baseline_deviation_pct': float(baseline_dev),
                'type_residential_small': int(ctype == 'residential_small'),
                'type_residential_large': int(ctype == 'residential_large'),
                'type_small_commercial':  int(ctype == 'small_commercial'),
                'type_industrial_light':  int(ctype == 'industrial_light'),
            })
        feat_df = pd.DataFrame(rows).set_index('consumer_id').fillna(0.0)
        return feat_df[FEATURE_COLS]

print('Loading dataset readings (~1.8M rows) and engineering features...')
full_readings = pd.read_csv(READINGS_CSV, low_memory=False)
labels = pd.read_csv(LABELS_CSV)

if 'consumer_type' not in full_readings.columns and 'consumer_type' in labels.columns:
    full_readings['consumer_type'] = full_readings['consumer_id'].map(labels.set_index('consumer_id')['consumer_type'])

consumer_meta = labels.set_index('consumer_id')[['base_kwh']] if 'base_kwh' in labels.columns else None
X = engineer_features(full_readings, consumer_meta)
label_map = labels.set_index('consumer_id')['label']
y_str = X.index.map(label_map)
valid = y_str.notna()
X = X[valid]
y_str = y_str[valid]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.classes_ = np.array(CLASS_NAMES)
y = le.transform(y_str)

print(f'Feature matrix generated: {X.shape[0]} consumers × {X.shape[1]} features')
display(X.head())


## 6. XGBoost Multi-Class Training with SMOTE & 5-Fold Stratified CV
- **SMOTE** balances minority anomaly classes.
- **XGBoost Classifier** learns non-linear patterns across load profiles.


In [ ]:
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

XGB_PARAMS = {
    'n_estimators':      400,
    'max_depth':         6,
    'learning_rate':     0.08,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'min_child_weight':  3,
    'eval_metric':       'mlogloss',
    'random_state':      42,
    'tree_method':       'hist',
    'n_jobs':            -1,
}

print('Running Stratified 5-Fold Cross Validation...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accs, fold_f1s = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx],      y[val_idx]

    smote = SMOTE(random_state=42, k_neighbors=3)
    X_res, y_res = smote.fit_resample(X_tr, y_tr)

    clf = XGBClassifier(**XGB_PARAMS)
    clf.fit(X_res, y_res, eval_set=[(X_val, y_val)], verbose=False)

    preds = clf.predict(X_val)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')
    fold_accs.append(acc)
    fold_f1s.append(f1)
    print(f'  Fold {fold}/5 — Accuracy: {acc:.4f} | Macro-F1: {f1:.4f}')

mean_acc, mean_f1 = np.mean(fold_accs), np.mean(fold_f1s)
print('')
print(f'CV Mean Accuracy: {mean_acc:.4f} | CV Mean Macro-F1: {mean_f1:.4f}')
print('')
print('Training final XGBoost model on full dataset with SMOTE...')
smote = SMOTE(random_state=42, k_neighbors=3)
X_full_res, y_full_res = smote.fit_resample(X, y)
final_clf = XGBClassifier(**XGB_PARAMS)
final_clf.fit(X_full_res, y_full_res, verbose=False)
print('Final model trained successfully!')


## 7. Model Evaluation & Confusion Matrix Heatmap


In [ ]:
y_pred = final_clf.predict(X)
print('=== Full Dataset Classification Report ===')
print(classification_report(y, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
plt.title('Confusion Matrix — Multi-Class Anomaly Detection', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.show()


## 8. Explainable AI (XAI) with SHAP
Computes **TreeExplainer** values to extract global and local feature importance.


In [ ]:
import shap

print('Computing SHAP values using TreeExplainer...')
explainer = shap.TreeExplainer(final_clf)
shap_values = explainer.shap_values(X)

global_shap_dict = {}
if isinstance(shap_values, list):
    for i, cls in enumerate(CLASS_NAMES):
        mean_abs = np.abs(shap_values[i]).mean(axis=0)
        global_shap_dict[cls] = dict(zip(FEATURE_COLS, mean_abs.tolist()))
else:
    for i, cls in enumerate(CLASS_NAMES):
        mean_abs = np.abs(shap_values[:, :, i]).mean(axis=0)
        global_shap_dict[cls] = dict(zip(FEATURE_COLS, mean_abs.tolist()))

plt.figure(figsize=(10, 6))
theft_importance = pd.Series(global_shap_dict['Theft']).sort_values(ascending=True).tail(10)
theft_importance.plot(kind='barh', color='#ef4444')
plt.title("Top 10 Feature Importance Driving 'Electricity Theft' (SHAP)", fontweight='bold')
plt.xlabel('Mean |SHAP Value| (Impact on Model Output)')
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
fault_importance = pd.Series(global_shap_dict['FaultyMeter']).sort_values(ascending=True).tail(10)
fault_importance.plot(kind='barh', color='#f59e0b')
plt.title("Top 10 Feature Importance Driving 'Faulty Meter' (SHAP)", fontweight='bold')
plt.xlabel('Mean |SHAP Value| (Impact on Model Output)')
plt.grid(True, alpha=0.3)
plt.show()


## 9. Single-Consumer Diagnostic Explanation (Waterfall)


In [ ]:
theft_cids = labels[labels['label'] == 'Theft']['consumer_id'].values
sample_cid = theft_cids[0]
sample_x = X.loc[[sample_cid]]
sample_proba = final_clf.predict_proba(sample_x)[0]
pred_class_idx = int(sample_proba.argmax())
pred_label = CLASS_NAMES[pred_class_idx]

print(f'Consumer ID: {sample_cid}')
print(f'Predicted Diagnosis: {pred_label} (Confidence: {sample_proba[pred_class_idx]:.2%})')

if isinstance(shap_values, list):
    sample_sv = shap_values[pred_class_idx][X.index.get_loc(sample_cid)]
else:
    sample_sv = shap_values[X.index.get_loc(sample_cid), :, pred_class_idx]

contrib_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'value': sample_x.iloc[0].values,
    'shap_impact': sample_sv
}).sort_values(by='shap_impact', key=abs, ascending=False).head(8)

plt.figure(figsize=(10, 5))
bar_colors = ['#ef4444' if v > 0 else '#3b82f6' for v in contrib_df['shap_impact']]
plt.barh(contrib_df['feature'][::-1], contrib_df['shap_impact'][::-1], color=bar_colors[::-1])
plt.axvline(0, color='black', lw=0.8, linestyle='--')
plt.title(f'SHAP Feature Contribution for Consumer {sample_cid} ({pred_label})', fontweight='bold')
plt.xlabel('SHAP Impact (+ pushes towards anomaly, - pushes away)')
plt.grid(True, alpha=0.3)
plt.show()


## 10. Save Model Artifacts & Download
Saves model files to `services/ml/models/` and creates a download zip archive.


In [ ]:
import joblib, json, zipfile
from datetime import datetime, timezone

version_str = f"xgb-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
model_file = MODELS_DIR / f'model_{version_str}.joblib'
latest_file = MODELS_DIR / 'model_latest.joblib'
metrics_file = MODELS_DIR / f'metrics_{version_str}.json'
shap_file = MODELS_DIR / f'shap_{version_str}.json'

joblib.dump(final_clf, model_file)
joblib.dump(final_clf, latest_file)

rep = classification_report(y, y_pred, target_names=CLASS_NAMES, output_dict=True)
metrics_data = {
    'version': version_str,
    'accuracy': round(float(accuracy_score(y, y_pred)), 4),
    'macro_f1': round(float(f1_score(y, y_pred, average='macro')), 4),
    'cv_accuracy': round(float(mean_acc), 4),
    'cv_macro_f1': round(float(mean_f1), 4),
    'per_class': {
        cls: {
            'precision': round(rep[cls]['precision'], 4),
            'recall': round(rep[cls]['recall'], 4),
            'f1': round(rep[cls]['f1-score'], 4),
            'support': int(rep[cls]['support'])
        } for cls in CLASS_NAMES
    },
    'confusion_matrix': cm.tolist(),
    'training_samples': len(X),
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'feature_count': len(FEATURE_COLS)
}

with open(metrics_file, 'w') as f: json.dump(metrics_data, f, indent=2)
with open(MODELS_DIR / 'metrics_latest.json', 'w') as f: json.dump(metrics_data, f, indent=2)
with open(shap_file, 'w') as f: json.dump(global_shap_dict, f, indent=2)
with open(MODELS_DIR / 'shap_latest.json', 'w') as f: json.dump(global_shap_dict, f, indent=2)

print(f'Saved model artifacts to {MODELS_DIR}')

zip_path = REPO_ROOT / 'powerblind_models.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(latest_file, arcname='model_latest.joblib')
    zf.write(metrics_file, arcname='metrics.json')
    zf.write(shap_file, arcname='shap.json')

print(f'Created model archive: {zip_path}')

try:
    from google.colab import files
    print('Initiating archive download...')
    files.download(str(zip_path))
except Exception:
    print(f'File ready for download at: {zip_path}')


## 11. Run Live FastAPI ML Microservice in Background
Starts the FastAPI server on port 8000 and tests the `/health` endpoint.


In [ ]:
import subprocess, time, requests

print('Starting FastAPI ML service on port 8000 in background...')
ml_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=str(REPO_ROOT / 'services' / 'ml') if (REPO_ROOT / 'services' / 'ml').exists() else str(REPO_ROOT)
)

time.sleep(3)

try:
    resp = requests.get('http://127.0.0.1:8000/health', timeout=5)
    print('FastAPI Health Response:')
    print(resp.json())
except Exception as e:
    print('Health check response:', e)


## 12. Interactive Anomaly Diagnosis Widget
Select any consumer from the dataset to test anomaly diagnosis in real time!


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

df_labels = labels_df if 'labels_df' in globals() else labels
consumer_options = [(f"{cid} ({df_labels[df_labels['consumer_id']==cid]['label'].iloc[0]})", cid)
                    for cid in df_labels['consumer_id'].head(50)]

consumer_dropdown = widgets.Dropdown(
    options=consumer_options,
    value=consumer_options[0][1],
    description='Consumer:',
    layout=widgets.Layout(width='400px')
)

diagnose_button = widgets.Button(
    description='⚡ Diagnose Anomaly',
    button_style='primary',
    tooltip='Run XGBoost + SHAP inference'
)

output_area = widgets.Output()

def on_diagnose_clicked(b):
    with output_area:
        output_area.clear_output()
        cid = consumer_dropdown.value
        if cid in X.index:
            feat_row = X.loc[[cid]]
            proba = final_clf.predict_proba(feat_row)[0]
            top_class = CLASS_NAMES[int(proba.argmax())]
            conf = proba[int(proba.argmax())]
            gt_label = df_labels[df_labels['consumer_id']==cid]['label'].iloc[0]
            gt_sub = df_labels[df_labels['consumer_id']==cid]['anomaly_subtype'].iloc[0]

            color = '#ef4444' if top_class == 'Theft' else '#f59e0b' if top_class == 'FaultyMeter' else '#3b82f6' if top_class == 'CommFailure' else '#10b981'

            prob_items = ''.join([f'<li><b>{c}:</b> {p:.2%}</li>' for c, p in zip(CLASS_NAMES, proba)])
            html_output = f'''
            <div style="padding: 16px; border: 2px solid {color}; border-radius: 8px; background-color: #f8fafc; font-family: sans-serif; max-width: 600px;">
                <h3 style="margin: 0; color: {color};">Diagnosis: {top_class} ({conf:.1%} Confidence)</h3>
                <p><b>Consumer ID:</b> {cid} | <b>Ground Truth:</b> {gt_label} ({gt_sub})</p>
                <div style="background-color: #e2e8f0; border-radius: 4px; overflow: hidden; height: 16px; margin: 8px 0;">
                    <div style="width: {conf*100}%; background-color: {color}; height: 100%;"></div>
                </div>
                <h4>Class Probabilities:</h4>
                <ul>{prob_items}</ul>
            </div>
            '''
            display(HTML(html_output))
        else:
            print('Consumer not found in feature matrix.')

diagnose_button.on_click(on_diagnose_clicked)
display(widgets.VBox([consumer_dropdown, diagnose_button, output_area]))
